In [2]:
%pip install requests beautifulsoup4 networkx

from collections import deque
from urllib.parse import urljoin, urlsplit, urlunsplit, urldefrag
import time
import requests
from bs4 import BeautifulSoup
import networkx as nx

SEED_URLS = ["https://example.com/"]  
MAX_PAGES = 100
REQUEST_DELAY_SECONDS = 1.0
REQUEST_TIMEOUT_SECONDS = 15
USER_AGENT = "EducationalPageRankCrawler/1.0 (contact: replace-with-your-email@example.com)"

# Keep crawling within the seed hostnames. Set False only when you intend to crawl linked external domains too.
SAME_DOMAIN_ONLY = True

if not SEED_URLS:
    raise ValueError("Add at least one seed URL to SEED_URLS.")



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\iamwa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
SKIP_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".gif", ".svg", ".webp", ".ico",
    ".pdf", ".zip", ".gz", ".mp3", ".mp4", ".webm",
    ".css", ".js", ".xml", ".json", ".woff", ".woff2",
)

def normalize_url(url):
    url, _fragment = urldefrag(url.strip())
    parts = urlsplit(url)
    scheme = parts.scheme.lower()
    if scheme not in ("http", "https") or not parts.hostname:
        return None
    host = parts.hostname.lower()
    netloc = host if parts.port is None else f"{host}:{parts.port}"
    path = parts.path or "/"
    return urlunsplit((scheme, netloc, path, parts.query, ""))

def is_html_candidate(url):
    path = urlsplit(url).path.lower()
    return not path.endswith(SKIP_EXTENSIONS)

def hostname(url):
    return (urlsplit(url).hostname or "").lower()

seeds = [normalize_url(url) for url in SEED_URLS]
seeds = [url for url in seeds if url]
if not seeds:
    raise ValueError("No valid HTTP(S) seed URLs were provided.")
allowed_hosts = {hostname(url) for url in seeds}
print(f"Starting from {len(seeds)} valid seed URL(s): {seeds}")

Starting from 1 valid seed URL(s): ['https://example.com/']


In [4]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT, "Accept": "text/html,application/xhtml+xml"})
queue = deque(seeds)
queued = set(seeds)
visited = set()
page_links = {}  # source URL -> set of discovered destination URLs
fetch_errors = {}
last_request_at = 0.0

while queue and len(visited) < MAX_PAGES:
    url = queue.popleft()
    if url in visited:
        continue
    visited.add(url)

    wait = REQUEST_DELAY_SECONDS - (time.monotonic() - last_request_at)
    if wait > 0:
        time.sleep(wait)
    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT_SECONDS, allow_redirects=True)
        last_request_at = time.monotonic()
        response.raise_for_status()
        final_url = normalize_url(response.url) or url
        content_type = response.headers.get("Content-Type", "").lower()
        if "html" not in content_type:
            fetch_errors[url] = f"Skipped non-HTML response ({content_type or 'unknown content type'})"
            continue
        # Use the final URL after redirects as the node identity.
        visited.add(final_url)
        soup = BeautifulSoup(response.text, "html.parser")
        destinations = set()
        for tag in soup.find_all("a", href=True):
            absolute = normalize_url(urljoin(final_url, tag["href"]))
            if not absolute or not is_html_candidate(absolute):
                continue
            if SAME_DOMAIN_ONLY and hostname(absolute) not in allowed_hosts:
                continue
            destinations.add(absolute)
            if absolute not in visited and absolute not in queued and len(visited) + len(queue) < MAX_PAGES:
                queue.append(absolute)
                queued.add(absolute)
        page_links[final_url] = destinations
        print(f"Fetched {final_url} — found {len(destinations)} candidate link(s)")
    except requests.RequestException as exc:
        fetch_errors[url] = str(exc)
        print(f"Could not fetch {url}: {exc}")

print(f"\nFetched {len(page_links)} HTML page(s); {len(fetch_errors)} fetch/skip issue(s).")

Fetched https://example.com/ — found 0 candidate link(s)

Fetched 1 HTML page(s); 0 fetch/skip issue(s).


In [5]:
graph = nx.DiGraph()
for source, destinations in page_links.items():
    graph.add_node(source)
    for destination in destinations:
        # Only include an edge when the destination itself was successfully fetched.
        if destination in page_links:
            graph.add_edge(source, destination)

if graph.number_of_nodes() == 0:
    raise RuntimeError("The crawl produced no HTML pages. Check the seed URL, network access, and site response.")

pagerank = nx.pagerank(graph, alpha=0.85, max_iter=100, tol=1.0e-8)
ranked_pages = sorted(pagerank.items(), key=lambda item: item[1], reverse=True)
print(f"Graph: {graph.number_of_nodes()} pages, {graph.number_of_edges()} links")
print("\nTop pages by PageRank:")
for rank, (url, score) in enumerate(ranked_pages[:20], start=1):
    print(f"{rank:>2}. {score:.6f}  {url}")

Graph: 1 pages, 0 links

Top pages by PageRank:
 1. 1.000000  https://example.com/


In [6]:
import csv

with open("pagerank_scores.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["url", "pagerank"])
    writer.writerows(ranked_pages)

with open("link_edges.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["source", "destination"])
    writer.writerows(graph.edges())

print("Saved pagerank_scores.csv and link_edges.csv")

Saved pagerank_scores.csv and link_edges.csv
